# Patterns demo (synthetic data only)
How to call each helper in `patterns.py`. Replace the dummy layers with the challenge data at the event. No challenge data or results here.

In [1]:
import sys; sys.path.insert(0, '.')
import numpy as np, geopandas as gpd, rasterio, tempfile, os
from rasterio.transform import from_origin
from shapely.geometry import box, Point
import patterns as p
tmp = tempfile.mkdtemp(); X0, Y0 = 4_000_000, 3_000_000
def raster(name, arr, nodata=None):
    path = os.path.join(tmp, name)
    with rasterio.open(path, 'w', driver='GTiff', height=arr.shape[0], width=arr.shape[1], count=1,
                       dtype=arr.dtype, crs='EPSG:3035', transform=from_origin(X0, Y0, 100, 100), nodata=nodata) as d:
        d.write(arr, 1)
    return path
pop = raster('pop.tif', np.full((10, 10), 50, 'float32'))
depth = np.zeros((10, 10), 'float32'); depth[:, :5] = 1.0
model = raster('depth.tif', depth, nodata=-9999)
flood = gpd.GeoDataFrame(geometry=[box(X0 + 250, Y0 - 1000, X0 + 800, Y0)], crs='EPSG:3035')
homes = gpd.GeoDataFrame({'use': ['residential', 'other', 'residential']},
                         geometry=[Point(X0 + 300, Y0 - 50), Point(X0 + 600, Y0 - 500), Point(X0 + 950, Y0 - 50)], crs='EPSG:3035')

## 1. Check CRSs before any overlay

In [2]:
p.check_crs(flood=flood, homes=homes, population=pop, model=model)

,layer,crs,geographic
0,flood,EPSG:3035,False
1,homes,EPSG:3035,False
2,population,EPSG:3035,False
3,model,EPSG:3035,False


## 2. People in the flooded area (partly flooded cells weighted by flooded share)

In [3]:
total, cells = p.area_weighted_population(pop, flood)
print(f'weighted: {total:.0f} people; any-cell-touched upper bound: {cells.population.sum():.0f}')

weighted: 2750 people; any-cell-touched upper bound: 3000


## 3. Buildings inside the flood, by use

In [4]:
p.count_buildings(homes, flood, by='use')

,use,buildings
0,other,1
1,residential,1


## 4. Observed vs modelled: did the model expect this flood?

In [5]:
p.overlap_stats(flood, model, threshold=0.0)

{'hits': 20,
 'misses': 30,
 'false_alarms': 0,
 'hits_km2': 0.19999999999999998,
 'misses_km2': 0.30000000000000004,
 'false_alarms_km2': 0.0,
 'hit_rate': 0.4,
 'false_alarm_ratio': 0.0,
 'csi': 0.4}

## 5. Send a layer to the policy-tool map
`p.export_to_template(flood, 'observed_flood', '../template', columns=[])` then run `python3 tools/build_layers.py` in the template.